# Business value (Step 10)
Read-only. Computes every $ and KPI number stated in README.md. 100% synthetic data.

* Thresholds come from MLflow run `d89beaca8f81447780da60ed2ea7c811` (fraud_risk_model@champion), not hardcoded.
* "governed view" = `fe_bar_lakebase.fraud_dash.*` scope, which excludes amount > 10000 for everyone.
* "full synthetic set" = `workshop_guy_catalog.fe_bar_a0.scored_transactions` joined to `txn_features_gold` (needs pii).
* Scaled numbers in B.4 are illustrative, not measured.
* Output text is copied verbatim into `logs/business_value_2026-11-06.md`.

In [0]:
# Setup: identity, MLflow thresholds, data span
import mlflow, json, re, os, statistics
import pandas as pd

RUN_ID = "d89beaca8f81447780da60ed2ea7c811"
REPO = os.path.dirname(os.getcwd())  # notebook lives in validation_notebooks/

who = spark.sql("SELECT current_user() u, is_account_group_member('pii') pii").first()
print(f"current_user = {who.u}\nis_account_group_member('pii') = {who.pii}")
assert who.pii, "STOP: not in pii group; the full synthetic set would be filtered"

params = mlflow.get_run(RUN_ID).data.params
BLOCK, REVIEW = float(params["block_threshold"]), float(params["review_threshold"])
print(f"MLflow run {RUN_ID}: block_threshold = {BLOCK:.6f}, review_threshold = {REVIEW:.6f}")

span = spark.sql("""SELECT min(txn_ts) lo, max(txn_ts) hi, months_between(max(txn_ts), min(txn_ts)) m
                    FROM workshop_guy_catalog.fe_bar_a0.scored_transactions""").first()
MONTHS = float(span.m)
print(f"data span {span.lo} to {span.hi} = {MONTHS:.3f} months")

LOG = []  # markdown blocks for logs/business_value_<TODAY>.md
def md(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    out += ["| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(out)
def show(title, df):
    block = f"### {title}\n\n{md(df)}\n"
    LOG.append(block); print(block)
usd = lambda x: f"-${-float(x):,.2f}" if float(x) < 0 else f"${float(x):,.2f}"
pct = lambda x: f"{100*x:.2f}%"

{"ts": "2026-11-06 23:53:38.816", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTg4MzQ1OTg4ODA4NDMwMRABIAEyJDAxYTFiMzQ2LThlYmMtN2E4Yy05YTQzLTc4N2M5ZGEyOGM1ODokNmU0ZDA3NjAtZDhmYS0zMWI5LWIzMDktNmMwYmZlYTY3ZmIxSgsIlNC51wYQgL6xSlABWAFgAWiuuJuPnMWjDYgBAA==.", "context": {}}
{"ts": "2026-11-06 23:53:38.816", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTg4MzQ1OTg4ODA4NDMwMRABIAEyJDAxYTFiMzQ2LThlYmMtN2E4Yy05YTQzLTc4N2M5ZGEyOGM1ODokNmU0ZDA3NjAtZDhmYS0zMWI5LWIzMDktNmMwYmZlYTY3ZmIxSgsIlNC51wYQgL6xSlABWAFgAWiuuJuPnMWjDYgBAA==.", "context": {}}
{"ts": "2026-11-06 23:53:38.816", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTg4MzQ1OTg4ODA4NDMwMRABIAEyJDAxYTFiMzQ2LThlYmMtN2E4Yy05YTQzLTc4N2M5ZGEyOGM1ODokNmU0ZDA3NjAtZDhmYS0zMWI5LWIzMDktNmMwYmZl

current_user = alex.owen@databricks.com
is_account_group_member('pii') = True
MLflow run d89beaca8f81447780da60ed2ea7c811: block_threshold = 0.950267, review_threshold = 0.871416
data span 2026-04-20 00:06:09 to 2026-10-16 23:55:43 = 5.903 months


In [0]:
# B.1 baseline_comparison and exposure through UC (governed view, amount <= 10000)
bc = spark.table("fe_bar_lakebase.fraud_dash.baseline_comparison").toPandas()
ex = spark.table("fe_bar_lakebase.fraud_dash.exposure").orderBy("recommended_action", "is_fraud").toPandas()
print("fe_bar_lakebase.fraud_dash.baseline_comparison")
print(bc.T.to_string(header=False))
print("\nfe_bar_lakebase.fraud_dash.exposure")
print(ex.to_string(index=False))

fe_bar_lakebase.fraud_dash.baseline_comparison
strategy                                          baseline_rule                                  model
rule_text             is_cross_border = 1 AND is_new_device = 1  recommended_action IN (block, review)
rows_evaluated                                           463881                                 463881
flagged_count                                             33723                                  24201
fraud_caught_count                                         2767                                   6571
fraud_amount_caught                                   933889.13                             7010482.03
false_positive_count                                      30956                                  17630
fraud_missed_count                                         7560                                   3756
fraud_amount_missed                                  7052318.79                              975725.89
total_fraud_amount        

In [0]:
# B.2 Model vs baseline rule, recomputed with MLflow thresholds
spark.sql(f"""CREATE OR REPLACE TEMP VIEW bv AS
SELECT s.txn_id, s.amount, s.recommended_action,
  CASE WHEN s.risk_score >= {BLOCK} THEN 'block' WHEN s.risk_score >= {REVIEW} THEN 'review' ELSE 'approve' END AS band,
  g.is_fraud, (g.is_cross_border = 1 AND g.is_new_device = 1) AS rule_flag
FROM workshop_guy_catalog.fe_bar_a0.scored_transactions s
JOIN workshop_guy_catalog.fe_bar_a0.txn_features_gold g USING (txn_id)""")
chk = spark.sql("SELECT count(*) n, sum(int(band <> recommended_action)) mismatch FROM bv").first()
print(f"rows joined = {chk.n}; band from MLflow thresholds vs stored recommended_action mismatches = {chk.mismatch}")

def scope_stats(where):
    return spark.sql(f"""SELECT count(*) rows, sum(int(NOT is_fraud)) legit,
      sum(int(band IN ('block','review'))) m_flag, sum(int(band IN ('block','review') AND is_fraud)) m_tp,
      sum(int(band IN ('block','review') AND NOT is_fraud)) m_fp,
      coalesce(sum(CASE WHEN band IN ('block','review') AND is_fraud THEN amount END),0) m_caught,
      coalesce(sum(CASE WHEN band = 'approve' AND is_fraud THEN amount END),0) m_missed,
      sum(int(rule_flag)) r_flag, sum(int(rule_flag AND is_fraud)) r_tp, sum(int(rule_flag AND NOT is_fraud)) r_fp,
      coalesce(sum(CASE WHEN rule_flag AND is_fraud THEN amount END),0) r_caught,
      coalesce(sum(CASE WHEN NOT rule_flag AND is_fraud THEN amount END),0) r_missed
      FROM bv {where}""").first().asDict()

S = {"governed": scope_stats("WHERE amount <= 10000"), "full": scope_stats(""), "hv": scope_stats("WHERE amount > 10000")}
g = S["governed"]

# cross-check against the view (independent source)
v = bc.set_index("strategy")
for col, key in [("fraud_amount_caught","caught"),("false_positive_count","fp"),("fraud_amount_missed","missed"),("flagged_count","flag")]:
    ok = all(abs(float(v.loc[s, col]) - float(g[p+key])) < 0.01 for s, p in [("model","m_"),("baseline_rule","r_")])
    print(f"cross-check {col}: recomputed == fraud_dash.baseline_comparison -> {ok}")

def compare(st):
    rows = [
        ("Transactions flagged", f"{st['r_flag']:,}", f"{st['m_flag']:,}", f"{st['m_flag']-st['r_flag']:+,}"),
        ("Fraud transactions caught", f"{st['r_tp']:,}", f"{st['m_tp']:,}", f"{st['m_tp']-st['r_tp']:+,}"),
        ("Fraud $ caught", usd(st['r_caught']), usd(st['m_caught']), usd(float(st['m_caught'])-float(st['r_caught']))),
        ("Fraud $ missed", usd(st['r_missed']), usd(st['m_missed']), usd(float(st['m_missed'])-float(st['r_missed']))),
        ("False positives (count)", f"{st['r_fp']:,}", f"{st['m_fp']:,}", f"{st['m_fp']-st['r_fp']:+,}"),
        ("False positive rate (FP / legitimate txns)", pct(st['r_fp']/st['legit']), pct(st['m_fp']/st['legit']), f"{100*(st['m_fp']-st['r_fp'])/st['legit']:+.2f} pts"),
        ("Precision of flags (fraud / flagged)", pct(st['r_tp']/st['r_flag']), pct(st['m_tp']/st['m_flag']), f"{100*(st['m_tp']/st['m_flag']-st['r_tp']/st['r_flag']):+.2f} pts"),
    ]
    return pd.DataFrame(rows, columns=["Metric", "Baseline rule (cross_border AND new_device)", "Model (block + review)", "Difference"])

show(f"B.2 Model vs baseline rule, governed view (excludes amount > 10000), {g['rows']:,} txns", compare(g))

rows joined = 464632; band from MLflow thresholds vs stored recommended_action mismatches = 0
cross-check fraud_amount_caught: recomputed == fraud_dash.baseline_comparison -> True
cross-check false_positive_count: recomputed == fraud_dash.baseline_comparison -> True
cross-check fraud_amount_missed: recomputed == fraud_dash.baseline_comparison -> True
cross-check flagged_count: recomputed == fraud_dash.baseline_comparison -> True
### B.2 Model vs baseline rule, governed view (excludes amount > 10000), 463,881 txns

| Metric | Baseline rule (cross_border AND new_device) | Model (block + review) | Difference |
|---|---|---|---|
| Transactions flagged | 33,723 | 24,201 | -9,522 |
| Fraud transactions caught | 2,767 | 6,571 | +3,804 |
| Fraud $ caught | $933,889.13 | $7,010,482.03 | $6,076,592.90 |
| Fraud $ missed | $7,052,318.79 | $975,725.89 | -$6,076,592.90 |
| False positives (count) | 30,956 | 17,630 | -13,326 |
| False positive rate (FP / legitimate txns) | 6.83% | 3.89% | -2.94 pts 

In [0]:
# B.3 Analyst time and measured agent latency
MAN_LO, MAN_HI, AGT_LO, AGT_HI = 15, 20, 2, 3  # minutes per case, ASSUMPTION from the original plan
m_cases = g["m_flag"] / MONTHS
r_cases = g["r_flag"] / MONTHS
hrs = lambda n, lo, hi: f"{n*lo/60:,.0f} to {n*hi/60:,.0f}"
show("B.3a Analyst hours per month, governed view (assumption: 15 to 20 min manual, 2 to 3 min with agent)", pd.DataFrame([
    ("Baseline rule queue, manual review", f"{r_cases:,.0f}", "15 to 20 (assumption)", hrs(r_cases, MAN_LO, MAN_HI)),
    ("Model queue (block + review), manual review", f"{m_cases:,.0f}", "15 to 20 (assumption)", hrs(m_cases, MAN_LO, MAN_HI)),
    ("Model queue (block + review), with agent", f"{m_cases:,.0f}", "2 to 3 (assumption)", hrs(m_cases, AGT_LO, AGT_HI)),
], columns=["Queue", "Cases per month (measured / 5.90 months)".replace("5.90", f"{MONTHS:.2f}"), "Minutes per case", "Analyst hours per month"]))

# measured end-to-end agent latency from committed MLflow trace exports
runs = []
t1 = json.load(open(f"{REPO}/logs/agent_trace_2026-10-21.json"))
runs += [("logs/agent_trace_2026-10-21.json", k, v["trace_id"], v["duration_ms"]) for k, v in t1.items()]
t2 = json.load(open(f"{REPO}/logs/trace_evidence_2026-11-01.json"))
runs += [("logs/trace_evidence_2026-11-01.json", "trace", t["trace_id"], t["execution_duration_ms"]) for t in t2["traces"]]
lat = pd.DataFrame(runs, columns=["source", "run", "trace_id", "duration_ms"])
print(lat.to_string(index=False))
secs = [d/1000 for d in lat.duration_ms]
AGENT_MED, AGENT_MIN, AGENT_MAX = statistics.median(secs), min(secs), max(secs)
show("B.3b Measured agent end-to-end latency vs assumption", pd.DataFrame([
    ("Agent run, end to end (explain, act, guardrail)", len(secs), f"{AGENT_MED:.1f} s", f"{AGENT_MIN:.1f} to {AGENT_MAX:.1f} s", "2 to 3 min per case (assumption includes analyst reading time)"),
], columns=["Measure", "Runs", "Median", "Range", "Assumption it sits next to"]))

lb = open(f"{REPO}/logs/lakebase_scores_sync_2026-10-20.md").read()
lb_lines = [l.strip() for l in lb.splitlines() if re.search(r"p95", l, re.I)]
print("Lakebase score lookup latency, logs/lakebase_scores_sync_2026-10-20.md:")
for l in lb_lines: print("  ", l)

### B.3a Analyst hours per month, governed view (assumption: 15 to 20 min manual, 2 to 3 min with agent)

| Queue | Cases per month (measured / 5.90 months) | Minutes per case | Analyst hours per month |
|---|---|---|---|
| Baseline rule queue, manual review | 5,713 | 15 to 20 (assumption) | 1,428 to 1,904 |
| Model queue (block + review), manual review | 4,100 | 15 to 20 (assumption) | 1,025 to 1,367 |
| Model queue (block + review), with agent | 4,100 | 2 to 3 (assumption) | 137 to 205 |

                             source                    run                            trace_id  duration_ms
   logs/agent_trace_2026-10-21.json            explain_run tr-d189164333a825572d0670df21c319ea        44100
   logs/agent_trace_2026-10-21.json                act_run tr-2fb3e5d6190f61225d050ad0d35e23d8        43400
   logs/agent_trace_2026-10-21.json        idempotency_run tr-135952fdfb4a0b099e719fc72d813a4f        41917
   logs/agent_trace_2026-10-21.json      guardrail_unknown tr-ceb9e85fc5

In [0]:
# B.4 Illustrative issuer scaling (linear, NOT measured)
EXAMPLE_TXNS_PER_MONTH = 10_000_000  # ASSUMPTION: example issuer volume
f = S["full"]
syn_per_month = f["rows"] / MONTHS
k = EXAMPLE_TXNS_PER_MONTH / syn_per_month
sc = lambda x: float(x) / MONTHS * k
show(f"B.4 Illustrative issuer at {EXAMPLE_TXNS_PER_MONTH:,} card txns/month (scale factor {k:.1f}x, full synthetic set)", pd.DataFrame([
    ("Card transactions per month", f"{syn_per_month:,.0f}", f"{EXAMPLE_TXNS_PER_MONTH:,}", "Assumption: example volume"),
    ("Model flags per month", f"{f['m_flag']/MONTHS:,.0f}", f"{sc(f['m_flag']):,.0f}", "Assumption: same flag rate as synthetic set"),
    ("Baseline rule flags per month", f"{f['r_flag']/MONTHS:,.0f}", f"{sc(f['r_flag']):,.0f}", "Assumption: same flag rate as synthetic set"),
    ("Fraud $ caught per month, model", usd(f['m_caught']/MONTHS), usd(sc(f['m_caught'])), "Assumption: same fraud rate and amount mix"),
    ("Fraud $ caught per month, baseline rule", usd(f['r_caught']/MONTHS), usd(sc(f['r_caught'])), "Assumption: same fraud rate and amount mix"),
    ("Fraud $ missed per month, model", usd(f['m_missed']/MONTHS), usd(sc(f['m_missed'])), "Assumption: same fraud rate and amount mix"),
    ("False positives per month, model", f"{f['m_fp']/MONTHS:,.0f}", f"{sc(f['m_fp']):,.0f}", "Assumption: same legitimate mix"),
    ("False positives per month, baseline rule", f"{f['r_fp']/MONTHS:,.0f}", f"{sc(f['r_fp']):,.0f}", "Assumption: same legitimate mix"),
    ("Analyst hours per month, rule queue, manual", hrs(f['r_flag']/MONTHS, MAN_LO, MAN_HI), hrs(sc(f['r_flag']), MAN_LO, MAN_HI), "Assumption: 15 to 20 min per case"),
    ("Analyst hours per month, model queue, with agent", hrs(f['m_flag']/MONTHS, AGT_LO, AGT_HI), hrs(sc(f['m_flag']), AGT_LO, AGT_HI), "Assumption: 2 to 3 min per case"),
], columns=["Row", "Synthetic set per month (measured)", "Illustrative issuer per month (scaled)", "Assumption on this row"]))

### B.4 Illustrative issuer at 10,000,000 card txns/month (scale factor 127.0x, full synthetic set)

| Row | Synthetic set per month (measured) | Illustrative issuer per month (scaled) | Assumption on this row |
|---|---|---|---|
| Card transactions per month | 78,711 | 10,000,000 | Assumption: example volume |
| Model flags per month | 4,219 | 536,037 | Assumption: same flag rate as synthetic set |
| Baseline rule flags per month | 5,721 | 726,833 | Assumption: same flag rate as synthetic set |
| Fraud $ caught per month, model | $1,795,566.52 | $228,120,640.64 | Assumption: same fraud rate and amount mix |
| Fraud $ caught per month, baseline rule | $188,839.86 | $23,991,464.21 | Assumption: same fraud rate and amount mix |
| Fraud $ missed per month, model | $181,520.53 | $23,061,567.86 | Assumption: same fraud rate and amount mix |
| False positives per month, model | 3,072 | 390,309 | Assumption: same legitimate mix |
| False positives per month, baseline rule | 5,250 | 667,044 | 

In [0]:
# B.4b High-value slice (amount > 10000) and headline KPIs in two scopes
pii = spark.sql("SELECT is_account_group_member('pii') pii").first().pii
print(f"is_account_group_member('pii') = {pii}")
assert pii, "STOP: must be pii to read the high-value slice"

h = S["hv"]
show("B.4b-1 High-value slice: amount > 10000, model flags (block or review), full synthetic set", pd.DataFrame([
    ("Transactions with amount > 10000", f"{h['rows']:,}"),
    ("Flagged by model (block or review)", f"{h['m_flag']:,}"),
    ("Fraud among flagged", f"{h['m_tp']:,}"),
    ("Fraud $ among flagged", usd(h['m_caught'])),
    ("False positives among flagged", f"{h['m_fp']:,}"),
    ("Visible in Genie / dashboard / App (fraud_dash views)", "0 (views filter amount > 10000 for everyone)"),
], columns=["Measure", "Value (scope: amount > 10000, full synthetic set)"]))

bands = spark.sql("""SELECT band, count(*) txns, sum(int(is_fraud)) fraud, round(coalesce(sum(CASE WHEN is_fraud THEN amount END),0),2) fraud_usd,
                     sum(int(NOT is_fraud)) legit FROM bv WHERE amount > 10000 GROUP BY band ORDER BY band""").toPandas()
show("B.4b-2 High-value slice by band (scope: amount > 10000)", bands)

head = lambda st: [
    ("Fraud $ caught, model", usd(st['m_caught'])), ("Fraud $ caught, baseline rule", usd(st['r_caught'])),
    ("Fraud $ missed, model", usd(st['m_missed'])), ("False positives, model", f"{st['m_fp']:,}"),
    ("False positives, baseline rule", f"{st['r_fp']:,}"),
    ("False positive rate, model", pct(st['m_fp']/st['legit'])), ("False positive rate, baseline rule", pct(st['r_fp']/st['legit'])),
]
hk = pd.DataFrame([(a, b, c) for (a, b), (_, c) in zip(head(g), head(f))],
                  columns=["Headline KPI", f"governed view (excludes amount > 10000), {g['rows']:,} txns", f"full synthetic set, {f['rows']:,} txns"])
show("B.4b-3 Headline KPIs in two scopes", hk)

is_account_group_member('pii') = True
### B.4b-1 High-value slice: amount > 10000, model flags (block or review), full synthetic set

| Measure | Value (scope: amount > 10000, full synthetic set) |
|---|---|
| Transactions with amount > 10000 | 751 |
| Flagged by model (block or review) | 705 |
| Fraud among flagged | 200 |
| Fraud $ among flagged | $3,588,732.92 |
| False positives among flagged | 505 |
| Visible in Genie / dashboard / App (fraud_dash views) | 0 (views filter amount > 10000 for everyone) |

### B.4b-2 High-value slice by band (scope: amount > 10000)

| band | txns | fraud | fraud_usd | legit |
|---|---|---|---|---|
| approve | 46 | 4 | 95788.35 | 42 |
| block | 71 | 21 | 394748.59 | 50 |
| review | 634 | 179 | 3193984.33 | 455 |

### B.4b-3 Headline KPIs in two scopes

| Headline KPI | governed view (excludes amount > 10000), 463,881 txns | full synthetic set, 464,632 txns |
|---|---|---|
| Fraud $ caught, model | $7,010,482.03 | $10,599,214.95 |
| Fraud $ caught, bas

In [0]:
# B.5 Final KPI table (used by README.md)
kpi = pd.DataFrame([
    ("Time to flag and explain a risky transaction", "Hours (assumption: manual queue, original plan)",
     f"Scored in batch; agent explains in {AGENT_MED:.1f} s median ({AGENT_MIN:.1f} to {AGENT_MAX:.1f} s, {len(secs)} runs)", "Agent traces", "B.3b, measured"),
    ("Analyst time per case", "15 to 20 min (assumption)", "2 to 3 min (assumption)", "Per case", "Original plan assumption"),
    ("Analyst hours per month", f"{hrs(r_cases, MAN_LO, MAN_HI)} (rule queue, manual)", f"{hrs(m_cases, AGT_LO, AGT_HI)} (model queue, with agent)", "Governed view", "B.3a, measured volume x assumed minutes"),
    ("False positive rate", pct(g['r_fp']/g['legit']), pct(g['m_fp']/g['legit']), "Governed view", "B.2, measured"),
    ("False positives (count)", f"{g['r_fp']:,}", f"{g['m_fp']:,}", "Governed view", "B.2, measured"),
    ("Fraud $ caught (flagged before approval)", usd(g['r_caught']), usd(g['m_caught']), "Governed view", "B.2, measured"),
    ("Fraud $ caught (flagged before approval)", usd(f['r_caught']), usd(f['m_caught']), "Full synthetic set", "B.4b-3, measured"),
    ("Fraud $ missed", usd(g['r_missed']), usd(g['m_missed']), "Governed view", "B.2, measured"),
    ("Fraud $ missed", usd(f['r_missed']), usd(f['m_missed']), "Full synthetic set", "B.4b-3, measured"),
    ("High-value flagged txns not triageable in Genie / dashboard / App", "n/a", f"{h['m_flag']:,} txns, {h['m_tp']:,} fraud, {usd(h['m_caught'])}", "amount > 10000", "B.4b-1, measured (open gap)"),
], columns=["KPI", "Baseline", "With the build", "Scope", "Source cell or assumption"])
show("B.5 Final KPI table", kpi)

### B.5 Final KPI table

| KPI | Baseline | With the build | Scope | Source cell or assumption |
|---|---|---|---|---|
| Time to flag and explain a risky transaction | Hours (assumption: manual queue, original plan) | Scored in batch; agent explains in 19.9 s median (16.0 to 51.1 s, 9 runs) | Agent traces | B.3b, measured |
| Analyst time per case | 15 to 20 min (assumption) | 2 to 3 min (assumption) | Per case | Original plan assumption |
| Analyst hours per month | 1,428 to 1,904 (rule queue, manual) | 137 to 205 (model queue, with agent) | Governed view | B.3a, measured volume x assumed minutes |
| False positive rate | 6.83% | 3.89% | Governed view | B.2, measured |
| False positives (count) | 30,956 | 17,630 | Governed view | B.2, measured |
| Fraud $ caught (flagged before approval) | $933,889.13 | $7,010,482.03 | Governed view | B.2, measured |
| Fraud $ caught (flagged before approval) | $1,114,720.20 | $10,599,214.95 | Full synthetic set | B.4b-3, measured |
| Fraud $ missed |

In [0]:
# B.6 Markdown for logs/business_value_2026-11-06.md (copied verbatim from this output)
text = "\n".join(LOG)
print(f"characters: {len(text)}")
print(text)

characters: 5283
### B.2 Model vs baseline rule, governed view (excludes amount > 10000), 463,881 txns

| Metric | Baseline rule (cross_border AND new_device) | Model (block + review) | Difference |
|---|---|---|---|
| Transactions flagged | 33,723 | 24,201 | -9,522 |
| Fraud transactions caught | 2,767 | 6,571 | +3,804 |
| Fraud $ caught | $933,889.13 | $7,010,482.03 | $6,076,592.90 |
| Fraud $ missed | $7,052,318.79 | $975,725.89 | -$6,076,592.90 |
| False positives (count) | 30,956 | 17,630 | -13,326 |
| False positive rate (FP / legitimate txns) | 6.83% | 3.89% | -2.94 pts |
| Precision of flags (fraud / flagged) | 8.21% | 27.15% | +18.95 pts |

### B.3a Analyst hours per month, governed view (assumption: 15 to 20 min manual, 2 to 3 min with agent)

| Queue | Cases per month (measured / 5.90 months) | Minutes per case | Analyst hours per month |
|---|---|---|---|
| Baseline rule queue, manual review | 5,713 | 15 to 20 (assumption) | 1,428 to 1,904 |
| Model queue (block + review), 